# 🎬 Audio-Automation: Chatterbox V3 GPU Studio & Autopilot Novel Publisher
### Channel: Garib YT (@garib_yt_official)
- **Autopilot Novel Rotation:** Automatically picks next unproduced 3-5+ Hours mega novel from archive; never repeats stories (`config/produced_stories.json`)
- **Full Cohesive Novel:** 100% complete story with zero skipped dialogues or artificial truncation
- **TTS Engine:** Chatterbox V3 Multilingual (GPU) with Step 11 Cinema Quality Radio DSP & Dubstudio CUDA flow clamp
- **Video Engine:** 1080p Full HD Video with Smartphone Music Player Center Scrolling Title (no background patti, pure white bold text, black shadow) + Glowing 16:9 Climax Thumbnail + Bottom Dialogue Subtitles + Cyan Audio Waveform + Progress Bar
- **Direct YouTube Upload:** Automated chunked upload to channel `Garib YT` via YouTube Data API v3


In [ ]:
# 1. Environment & GPU Setup
import os, sys, torch

assert torch.cuda.is_available(), '❌ GPU must be enabled in Kaggle Settings (Tesla T4 or P100)!'
print(f'🚀 Kaggle GPU Detected: {torch.cuda.get_device_name(0)}')

# Optimization for PyTorch CUDA memory allocation (Dubstudio standard)
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TTS_ENGINE'] = 'chatterbox'
os.environ['CHANNEL_NAME'] = 'Garib YT'

# API Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['GEMINI_API_KEY'] = user_secrets.get_secret('GEMINI_API_KEY')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        os.environ['GEMINI_API_KEY'] = 'AQ.Ab8RN6yQW47w5o4ZJqR1tM3sK7vP9bC2xL8eY0uN6dF1'

print('✅ Environment configured for Garib YT Drama Studio.')


In [ ]:
# 2. Install System & Audio/ML Dependencies (Python 3.13 Compatible Stack)
import subprocess, sys, os

print('📦 Installing system libraries & Hindi fonts...')
!sudo apt-get update -qq || true
!sudo apt-get install -y -qq ffmpeg sox libsox-dev build-essential python3-dev fonts-lohit-deva fonts-dejavu-core fonts-noto > /dev/null

print('📦 Upgrading build tools...')
!pip install -q --upgrade pip wheel setuptools

print('📦 Installing Chatterbox, Transformers & Audio Packages...')
!pip install -q --no-cache-dir conformer==0.3.2 pykakasi==2.3.0 pyloudnorm s3tokenizer diffusers==0.29.0 'safetensors>=0.8.0'
!pip install -q --no-cache-dir resemble-perth sox vocos einops spacy-pkuseg soundfile pydub requests beautifulsoup4 google-generativeai python-dotenv 'librosa>=0.10.0' edge-tts google-api-python-client google-auth-oauthlib google-auth-httplib2 Pillow
!pip install -q --no-cache-dir --no-deps git+https://github.com/resemble-ai/chatterbox.git

print('📦 Pinning stable NumPy 2.2.x & SciPy 1.15.x stack...')
!pip install -q --no-cache-dir 'numpy>=2.1.0,<2.3.0' 'scipy>=1.14.0,<1.16.0' 'safetensors>=0.8.0' 'transformers>=4.48.0'

# Python 3.12/3.13 Kaggle compatibility patch (Dubstudio fix)
import pkgutil, zipimport, importlib.machinery
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = zipimport.zipimporter
if hasattr(importlib.machinery, 'FileFinder'):
    importlib.machinery.FileFinder.find_module = lambda self, fullname, *args, **kwargs: self.find_spec(fullname).loader if self.find_spec(fullname) is not None else None

print('\n🔍 Verifying library imports...')
import numpy as np, scipy, scipy.signal, torch
print(f'  torch: {torch.__version__}, cuda: {torch.cuda.is_available()}')
print(f'  numpy: {np.__version__}')
print(f'  scipy: {scipy.__version__}')

from chatterbox.mtl_tts import ChatterboxMultilingualTTS
print('✅ Chatterbox Multilingual V3 successfully imported without errors!')


In [ ]:
# 3. Clone & Initialize Audio-Automation Studio with Garib YT Token
import os, sys, json, base64, shutil
from pathlib import Path

repo_dir = Path('/kaggle/working/Audio-automation')
if not repo_dir.exists():
    print('📥 Cloning latest Audio-automation repository from GitHub...')
    !git clone https://github.com/9329238475x-wq/Audio-automation.git /kaggle/working/Audio-automation
else:
    print('🔄 Updating existing repository...')
    !cd /kaggle/working/Audio-automation && git fetch origin && git reset --hard origin/main

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
os.chdir(str(repo_dir))

# Install Token for Channel: Garib YT (@garib_yt_official)
tokens_dir = repo_dir / 'tokens'
tokens_dir.mkdir(parents=True, exist_ok=True)

token_b64 = 'eyJwcm9maWxlIjogImdhcmliX3l0IiwgImNoYW5uZWxfaWQiOiAiVUNiN3VFV0ZReUM2SGNQaUtaR2lrTktnIiwgImNoYW5uZWxfdGl0bGUiOiAiR2FyaWIgWVQgIiwgImN1c3RvbV91cmwiOiAiQGdhcmliX3l0X29mZmljaWFsIiwgInRodW1ibmFpbCI6ICJodHRwczovL3l0My5nZ3BodC5jb20vQWY5ZXItMnlqcjhOUXBUWDdzNGgzWXNBNmlyNTFWa0YwaDhxWndEQWxmelJhVkRfcEVkMWxEakhnMGdoZzlsay1sRm5iSU9TenFzPXM4MDAtYy1rLWMweDAwZmZmZmZmLW5vLXJqIiwgInN1YnNjcmliZXJfY291bnQiOiAiOTUiLCAidmlkZW9fY291bnQiOiAiNiIsICJ0b2tlbiI6ICJ5YTI5LmEwQVgwN0NtdGRoM2VpN2hYZ0tyUXpkaUdSc04xbl8zYnNVTjQ0TThTXzV5LVNXUXJoSExaUnV1VkhJVjBDUFVaSDczeXVUeGZNamFpWExraE1CT29UVnNNRXlhYlZlSUVURHZlSTEzMkI2blRpVkVralBEcm02LWtpV1NYNXZ4YjB2TlkwQ0dQTXcyM2Y1YkNqemRrWjV3QnQyOW4wN1U5OTBYMzFGRnZvek9NRW1jRTNOdHZZdU5PbndrU0JQWmtMTFRRYWdBZ2FDZ1lLQVdBU0FSVVNGUUhHWDJNaXJzRERtcWVJVEwzY2g5SXVsWFZ4R2cwMjA2IiwgInJlZnJlc2hfdG9rZW4iOiAiMS8vMGc0d3pWeHdBcmwya0NnWUlBUkFBR0JBU053Ri1MOUlyWFBqZTk3ZmRueUZWSmJCV0o5aHFQdGE2dGhUMTZ2eVU4RVpKQS1nVjNCWnBlS056bkFoX0VIeFJmNTItS3ZoRkF3ZyIsICJ0b2tlbl91cmkiOiAiaHR0cHM6Ly9vYXV0aDIuZ29vZ2xlYXBpcy5jb20vdG9rZW4iLCAiY2xpZW50X2lkIjogIjE4ODQ4NDI5ODA4Ni11azd1aGR1dW04b2k5N2puZG9uaTIxaHRta3RrZWdvaC5hcHBzLmdvb2dsZXVzZXJjb250ZW50LmNvbSIsICJjbGllbnRfc2VjcmV0IjogIkdPQ1NQWC1KdzNBS0xhV0ZtWWdEeE9QUU5xdTNUb2pKYnduIiwgInNjb3BlcyI6IFsiaHR0cHM6Ly93d3cuZ29vZ2xlYXBpcy5jb20vYXV0aC95b3V0dWJlLnVwbG9hZCIsICJodHRwczovL3d3dy5nb29nbGVhcGlzLmNvbS9hdXRoL3lvdXR1YmUucmVhZG9ubHkiXX0='
garib_token_data = json.loads(base64.b64decode(token_b64.encode('ascii')).decode('utf-8'))

token_file = tokens_dir / 'token_garib_yt.json'
with open(token_file, 'w', encoding='utf-8') as f:
    json.dump(garib_token_data, f, indent=2)

active_channel_file = repo_dir / 'config' / 'active_channel.txt'
active_channel_file.parent.mkdir(parents=True, exist_ok=True)
active_channel_file.write_text('garib_yt', encoding='utf-8')

print('✅ Studio configured for Channel: Garib YT (@garib_yt_official)')
print(f'✅ Character Voice Samples: {len(list((repo_dir / "voice_samples").glob("*.wav")))} studio reference voices loaded!')


In [ ]:
# 4. Autopilot Drama Hunting: Select Next Unproduced Master Novel & Synthesize
import sys, os
from pathlib import Path

repo_dir = Path('/kaggle/working/Audio-automation')
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
os.chdir(str(repo_dir))

from story_engine.auto_drama_hunter import AutoDramaHunter
from pipeline import run_audio_story_pipeline

# Autopilot: Automatically checks produced_stories.json, picks next unproduced 3-5 hr novel
hunter = AutoDramaHunter()
drama = hunter.get_next_autopilot_drama(channel_name='Garib YT')

print('=' * 72)
print(f'🎬 SELECTED MASTER NOVEL: {drama["title"]}')
print(f'   Source File:  {drama["source_file"]}')
print(f'   Word Count:   {drama["word_count"]:,} words (~{drama["approx_hours"]} Hours continuous)')
print(f'   Total Scenes: {drama["total_scenes"]} complete scenes')
print(f'   Script Path:  {drama["script_path"]}')
print('=' * 72)

print('\n🎙️ Launching Chatterbox V3 Neural Audio & 1080p Video Pipeline on GPU...')
master_wav, master_mp3, master_mp4 = run_audio_story_pipeline(
    topic=drama['title'],
    hero_name=drama['hero_name'],
    heroine_name=drama['heroine_name'],
    custom_script_path=drama['script_path'],
    output_prefix=f"{drama['story_id']}_garib_yt",
    engine='chatterbox',
    render_video=True,
    channel_name='Garib YT'
)

print('\n' + '=' * 72)
print('🎉 MASTER 1080P MULTI-HOUR NOVEL VIDEO RENDERED SUCCESSFULLY!')
print(f'  Master MP4: {master_mp4}')
print('=' * 72)


In [ ]:
# 5. Direct Upload to YouTube Channel: Garib YT & Update Autopilot Database
import sys, os, json
from pathlib import Path

repo_dir = Path('/kaggle/working/Audio-automation')
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
os.chdir(str(repo_dir))

from uploader.youtube_uploader import YouTubeUploader
from story_engine.auto_drama_hunter import AutoDramaHunter
import soundfile as sf

print('=' * 72)
print('🚀 LAUNCHING AUTOMATIC YOUTUBE UPLOAD TO GARIB YT')
print('=' * 72)

video_path = Path(master_mp4)
assert video_path.exists(), f'Video file not found: {video_path}'

duration_s = sf.info(str(master_wav)).duration
dur_hours = round(duration_s / 3600.0, 2)

display_title = drama.get('display_title') or f"💔 {drama['title']} | Full Mega Novel Audio Story | Garib YT"
if len(display_title) > 98:
    display_title = display_title[:95] + '...'

desc = f'''सुनिए {drama['title']} - मुंशी प्रेमचंद का कालजयी शाहकार उपन्यास, संपूर्ण कहानी एक ही वीडियो में।

🎧 कृपया बेहतर अनुभव के लिए हेडफ़ोन (Headphones) का उपयोग करें।
🎙️ आवाज़: Chatterbox V3 Neural Audio Engine (Studio Mastered)
📺 चैनल: Garib YT (@garib_yt_official)

कहानी का सार:
यह कहानी है मानवीय भावनाओं, पारिवारिक त्याग, अनमेल विवाह और आत्मसम्मान की एक ऐसी दिल को छू लेने वाली गाथा जो आपकी आँखों में आँसू ला देगी।

#hindiromanticstories #hindiaudiobooks #premchand #nirmala #gaban #hindikahaniya #pocketfm #garibyt #audiobooks'''

tags = ['hindi novel', 'hindi audio book', 'munshi premchand', 'nirmala novel', 'gaban novel', 'hindi kahaniya', 'garib yt', 'pocket fm stories', 'audio story hindi']

# Locate thumbnail
thumb_path = video_path.parent / f'{video_path.stem}.jpg'
if not thumb_path.exists():
    possible_thumbs = list(video_path.parent.glob('*.jpg')) + list(Path('output/masters').glob('*.jpg'))
    thumb_path = possible_thumbs[0] if possible_thumbs else None

print(f'📺 Channel: Garib YT (@garib_yt_official)')
print(f'🎬 Video: {video_path.name} ({video_path.stat().st_size / (1024*1024):.2f} MB)')
print(f'⏱️ Duration: {duration_s:.1f}s ({dur_hours} Hours)')

uploader = YouTubeUploader(channel_profile='garib_yt')
ch_info = uploader.get_channel_info()
print(f'✅ Connected Channel: {ch_info.get("channel_title")} ({ch_info.get("custom_url")})')

upload_result = uploader.upload_video(
    video_path=video_path,
    title=display_title,
    description=desc,
    tags=tags,
    privacy_status='public',
    thumbnail_path=thumb_path
)

yt_id = upload_result.get('video_id')
yt_url = upload_result.get('url', f'https://youtu.be/{yt_id}')

print('\n' + '=' * 72)
print('🚀 PUBLISHED TO YOUTUBE SUCCESSFULLY!')
print(f'  Channel:     {ch_info.get("channel_title")} ({ch_info.get("custom_url")})')
print(f'  Video Title: {display_title}')
print(f'  Video ID:    {yt_id}')
print(f'  Live URL:    {yt_url}')
print(f'  Duration:    {dur_hours} Hours')
print('=' * 72)

# Update Autopilot database
hunter = AutoDramaHunter()
hunter.mark_story_completed(
    story_id=drama['story_id'],
    youtube_id=yt_id,
    youtube_url=yt_url,
    duration_seconds=duration_s
)
print('✅ Autopilot Database updated! Next run will automatically select the subsequent novel.')
